# SUT to IOT

Compare Eurostat methods A–D on MARIO's packaged SUT fixture.
See the [conversion guide](https://mario-suite.readthedocs.io/en/latest/user_guide/transformations/sut_to_iot.html) for the assumptions, examples and equations.

- **A:** product-by-product; a common input recipe for each product across producers.
- **B:** product-by-product; each industry allocates its average inputs across its outputs.
- **C:** industry-by-industry; a common customer mix for all products of an industry.
- **D:** industry-by-industry; a common customer mix for all producers of a product.

A/C need square, invertible supply and zero activity final demand (`Ya`); they can introduce negative entries. B/D also accept rectangular supply and nonzero `Ya`. This fixture supports all four methods. The fixture illustrates the API, not which assumption fits a real database.


In [ ]:
import mario
import pandas as pd

sut = mario.load_test("SUT")


## Inspect the starting SUT

Supply has activity rows and commodity columns; use has the reverse axes.


In [ ]:
{"supply": sut.S.shape, "use": sut.U.shape, "table": sut.table_type}


## Convert with B

Use `inplace=False` to retain the SUT. The returned IOT contains the converted baseline only; any other scenarios are omitted.


In [ ]:
iot = sut.to_iot(method="B", inplace=False)
iot.Z


In [ ]:
iot.get_index("Sector")


In [ ]:
iot.units["Sector"]


## Compare the four methods

A/B retain the commodity set; C/D retain the activity set. Compare dimensions, productive account totals and direct final-demand accounts together. Shapes and totals alone do not determine which assumption is appropriate.


In [ ]:
converted = {method: sut.to_iot(method=method, inplace=False) for method in "ABCD"}
pd.DataFrame([
    {
        "method": method,
        "sector_set": "Commodity" if method in "AB" else "Activity",
        "Z_shape": db.Z.shape,
        "negative_Z_entries": int((db.Z < 0).sum().sum()),
        "negative_Y_entries": int((db.Y < 0).sum().sum()),
    }
    for method, db in converted.items()
]).set_index("method")


In [ ]:
account_totals = {}
for prefix in ("E", "V"):
    before = sum(sut.query(prefix + suffix).sum(axis=1) for suffix in ("a", "c", "Y"))
    account_totals[prefix] = pd.DataFrame({
        "SUT": before,
        **{
            method: db.query(prefix).sum(axis=1) + db.query(prefix + "Y").sum(axis=1)
            for method, db in converted.items()
        },
    })
account_totals["E"]


In [ ]:
account_totals["V"]
